# Where do earthquakes and volcanoes happen — and why there?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2F04_where_and_why.ipynb).

Two files today. One holds every earthquake of magnitude 5.5 and above that the USGS has located
since the start of 2000. The other holds every volcano known to have erupted since the last ice
age. Put either one on a picture and something odd happens: the dots are not scattered. They fall
on lines — down the middle of oceans where there is no land at all, and round the rim of the
Pacific where there is almost nothing else.

You will draw those lines yourself, from nothing but longitude and latitude, and put the plate
boundaries on top of them. Then you will go one step further and colour each earthquake by how
deep it was. The deep ones are not where you would guess. Where they are is one of the clearest
pieces of evidence anybody has that the floor of the Pacific is sinking back into the mantle.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, then export the notebook as a PDF and upload that.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say where earthquakes and volcanoes happen, which kind of plate boundary each
one prefers, and where the deep earthquakes sit relative to a trench — and why that geometry is
what a sinking slab looks like. Read a count of eruptions off a scale where each step means ten
times more rock.

**The skills.** Draw a map with no map library: `plt.scatter(longitude, latitude)`, the coastline
and the plate boundaries from a CSV with `plt.plot`. Colour a scatter by a third column with `c=`
and `cmap=`, and label the colours with `plt.colorbar`. Put two panels side by side with
`plt.subplot`. And when the counts run from thousands down to single figures, `plt.yscale("log")`,
without which the figure hides most of its own data.

**Eight places where you write something: five in class, three at home.** Each one is headed
*Your turn*, with an empty cell under it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (9, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(url, cached):
    """Read the live source; fall back to the copy stored with the course."""
    try:
        return pd.read_csv(url)
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + cached)

USGS = ("https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&orderby=time-asc"
        "&starttime=2000-01-01&endtime=2026-01-01&minmagnitude=5.5")
GVP = ("https://webservices.volcano.si.edu/geoserver/GVP-VOTW/ows?service=WFS&version=1.0.0"
       "&request=GetFeature&typeName=GVP-VOTW:Smithsonian_VOTW_Holocene_")

quakes = load(USGS, "week04_2000-01-01_2026-01-01_M5.5.csv")
volcanoes = load(GVP + "Volcanoes&outputFormat=csv", "week04_gvp_volcanoes.csv")
eruptions = load(GVP + "Eruptions&outputFormat=csv", "week04_gvp_eruptions.csv")

# These two files live in this repository, so CACHE is their home rather than their fallback:
# there is no live server to try first.
coast = pd.read_csv(CACHE + "/coastlines.csv")
boundaries = pd.read_csv(CACHE + "/plate_boundaries.csv")

print("earthquakes:", quakes.shape, " volcanoes:", volcanoes.shape,
      " eruptions:", eruptions.shape, " boundary points:", boundaries.shape)

## Longitude across, latitude up

The earthquake catalogue has a `longitude` column and a `latitude` column. Longitude runs from
-180 to 180 across the world and latitude from -90 to 90 up it, so putting one on the bottom axis
and the other on the side is already a map. Nothing else is needed — no map package, no
projection, no install.

First, one line of housekeeping from the tables week: not every row in an earthquake catalogue is
an earthquake, so keep the rows where `type` says it is.

Then the map. Three things beyond the scatter earn their place, and they will be on every map you
draw this term:

- `plt.xlim` and `plt.ylim` fix the edges at the whole world, so one map is comparable with the
  next one;
- `plt.gca().set_aspect("equal")` makes one degree across the same length as one degree up, so
  nothing is stretched;
- the coastline goes on from `data/coastlines.csv` with a single `plt.plot`, exactly as in week
  one — the file has a blank row between coastline segments, which is what makes matplotlib lift
  the pen instead of joining Africa to Australia.

In [ ]:
quakes = quakes[quakes["type"] == "earthquake"]
print(len(quakes), "earthquakes")

In [ ]:
plt.scatter(quakes["longitude"], quakes["latitude"], s=1, color="crimson")
plt.plot(coast.lon, coast.lat, color="0.6", lw=0.6)
plt.xlim(-180, 180)
plt.ylim(-90, 90)
plt.gca().set_aspect("equal")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title(str(len(quakes)) + " earthquakes, M5.5 and above")
plt.savefig("earthquake_map.png", dpi=150)   # a copy you can drop into a report; before show()
plt.show()

The dots are not spread over the planet. They are lines, and two of them are worth staring at.

One runs round the edge of the Pacific, through Alaska, Japan, Indonesia, New Zealand and the
whole west coast of the Americas — the Ring of Fire, and if you had guessed anywhere before
running the cell, it was probably there. The other is the surprise: a line straight down the
middle of the Atlantic, from Iceland to the far south, thousands of kilometres from any coast. It
is drawn only by earthquakes. No land marks it, and the coastline you plotted underneath goes
nowhere near it.

Before the next map, one honest caveat about this one. Every degree of longitude is drawn the same
width, but on the ground a degree of longitude at latitude *L* is only `cos(L)` as wide as one at
the equator — the same `cos(latitude)` that came up when you weighted grid cells by area. So the
map stretches everything away from the equator sideways, by a factor of `1 / cos(L)`.

In [ ]:
for lat in [0, 30, 60, 72]:
    stretch = 1 / np.cos(np.deg2rad(lat))
    print("at latitude", lat, "this map stretches east-west by a factor of", round(stretch, 1))

At the latitude of Greenland that is a factor of 3.2, which is most of why
Greenland looks the size of Africa on maps like this one. Every flat map has to distort something;
this one keeps latitude and longitude honest as *coordinates* and pays for it in shape and area.
For finding out where things are, that is a fine trade, and it costs no extra library.

### ✏️ Your turn 1

The volcano table is loaded as `volcanoes`. Its columns are named differently from the earthquake
catalogue — `Longitude` and `Latitude`, with capitals — because it comes from a different archive.

Draw the same map for the volcanoes: a scatter of longitude against latitude, the coastline on
top, the same limits and aspect, labelled axes, and a title carrying how many there are. Use
`marker="^"` so the volcanoes are triangles, and `s=8` so they are big enough to see.

`plt.scatter` hands back the thing it drew. Catch it in a name — `triangles = plt.scatter(...)` —
so that the self-check can look at the points you actually plotted.

**Use these names**, because the self-check looks for them: `triangles`.

In [ ]:
# ← your answer here


assert len(triangles.get_offsets()) == len(volcanoes), \
    "the scatter drew something else — pass the volcano columns, not the earthquake ones"
assert round(triangles.get_offsets()[:, 1].max()) == round(volcanoes["Latitude"].max()), \
    "longitude goes across and latitude up — check which column you gave scatter first"
print("✓ the volcano map —", len(triangles.get_offsets()), "volcanoes, between latitude",
      round(triangles.get_offsets()[:, 1].min()), "and", round(triangles.get_offsets()[:, 1].max()))

## Which lines

The volcanoes fall on lines too, and mostly the same ones: the Andes, the Cascades, the Aleutians,
Japan, Indonesia. So both maps are drawing something neither file contains. That something is the
edges of the tectonic plates, and we can put them on the map from a third file.

`data/plate_boundaries.csv` was converted once from the USGS plate-boundary map into a plain table
of longitudes and latitudes. It has the blank-row trick that `coastlines.csv` has, so one
`plt.plot` draws a whole layer without joining the end of one line to the start of the next. It
also has a `kind` column, because there are three ways two plates can meet:

- a **ridge**, where they pull apart and new ocean floor is made;
- a **transform**, where they slide past each other;
- a **trench**, where one plate bends and goes down underneath the other.

In [ ]:
print(boundaries.head())
print(boundaries["kind"].value_counts())

In [ ]:
ridges = boundaries[boundaries["kind"] == "ridge"]
transforms = boundaries[boundaries["kind"] == "transform"]
trenches = boundaries[boundaries["kind"] == "trench"]

plt.scatter(quakes["longitude"], quakes["latitude"], s=1, color="0.75")
plt.plot(coast.lon, coast.lat, color="0.6", lw=0.5)
plt.plot(ridges.lon, ridges.lat, color="tab:blue", lw=1, label="ridge")
plt.plot(trenches.lon, trenches.lat, color="black", lw=1, label="trench")
plt.plot(transforms.lon, transforms.lat, color="tab:green", lw=1, label="transform")
plt.xlim(-180, 180)
plt.ylim(-90, 90)
plt.gca().set_aspect("equal")
plt.legend(loc="lower left")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title(str(len(quakes)) + " earthquakes and the plate boundaries")
plt.show()

The lines land on the dots. The blue ridge down the middle of the Atlantic is the line the
earthquakes drew on their own, and the black trenches trace the Ring of Fire. That is the answer
to the first half of today's question, and it is worth noticing how little work it took: two
scatter plots and three `plt.plot` calls.

One caution about the black line before we use it. The file calls seven of its features trenches
that are really continental collisions — the run of black across Asia through the Zagros and the
Himalaya is India and Arabia driving into Eurasia, not one plate sinking under another. Keep it in
mind for the next section, where the difference will be visible.

The volcanoes have their own opinion about which boundary they like, and the table will say so
directly. `Tectonic_Setting` is a text column, and `.str.startswith("Subduction")` asks the same
question of every row at once — a mask, like the ones you built on arrays, with *does this text
begin like that?* as the question.

In [ ]:
setting = volcanoes["Tectonic_Setting"]
n_subduction = setting.str.startswith("Subduction", na=False).sum()

print("subduction zone:", n_subduction)
print("rift zone:      ", setting.str.startswith("Rift", na=False).sum())
print("intraplate:     ", setting.str.startswith("Intraplate", na=False).sum())
print("subduction share:", round(100 * n_subduction / len(volcanoes)), "percent of",
      len(volcanoes), "volcanoes")

69% of them sit at a subduction zone. That is a mechanism, not a coincidence:
the plate going down carries wet ocean-floor minerals with it, and once it is deep enough those
minerals give their water up into the hot mantle above. Water lowers the melting temperature of
rock, so mantle that would otherwise stay solid melts, and the melt rises and builds a line of
volcanoes a hundred kilometres or so behind the trench. Ridges melt rock a different way — by
letting it rise and decompress — and they are the longest boundary system on the planet, but only
102 of these volcanoes sit on oceanic rift. Most ridge volcanism happens two
kilometres under water, where an eruption leaves nothing for anybody to write down. *A catalogue
lists what somebody's instruments recorded, not what happened. Where there are no seismometers
there are no earthquakes in the file.* The same is true of eruptions, and it will matter again
before the end of the notebook.

## How deep

Every earthquake in the catalogue has a `depth` in kilometres as well as a position. Deep
earthquakes are strange: at a few hundred kilometres down the rock is hot enough and squeezed hard
enough that it should flow rather than snap. Count them first, in three classes seismologists
actually use.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
quakes = quakes[quakes["type"] == "earthquake"]

In [ ]:
shallow = quakes[quakes["depth"] <= 70]
middle = quakes[(quakes["depth"] > 70) & (quakes["depth"] <= 300)]
deep = quakes[quakes["depth"] > 300]

print("shallow, 0-70 km:      ", len(shallow))
print("intermediate, 70-300:  ", len(middle))
print("deep, more than 300 km:", len(deep))
print("deepest:", quakes["depth"].max(), "km")

10,472 of the 12,847 are in the top 70 km, and the deepest is
691.6 km down. At that depth rock under that much pressure and heat deforms by flowing
rather than by breaking, so a deep earthquake is something that needs explaining. Where the
785 of them are is the question, and a map can answer it if the colour of each dot
carries the depth.

`plt.scatter` takes `c=` for the values to colour by and `cmap=` for the colour scheme;
`vmin` and `vmax` fix what the ends of the scale mean, so that the colours mean the same thing on
every map you draw. The thing `plt.scatter` hands back — the one you caught as `triangles` on the
volcano map — is what `plt.colorbar(label=...)` needs, to know which colours it is the key to.

### ✏️ Your turn 2

Redraw the world map of earthquakes, but colour each dot by its depth.

Pass `c=quakes["depth"]` and `cmap="plasma_r"` to `plt.scatter`, with `vmin=0` and `vmax=600` so
that the scale is fixed and pale means shallow. Catch what `plt.scatter` hands back, as you did on
the volcano map, and give that to `plt.colorbar`:

```
dots = plt.scatter(...)
plt.colorbar(dots, label="depth (km)")
```

Keep everything else from the first map — the coastline, the limits, the equal aspect, the labels,
and a title with the sample size in it.

**Use these names**, because the self-check looks for them: `dots`.

In [ ]:
# ← your answer here


assert dots.get_array() is not None, "pass c=quakes[\"depth\"], or the dots carry no colour"
assert dots.get_array().max() > 300, "the colours should carry depth in km, not magnitude"
print("✓ the depth map — the colours run from", round(quakes["depth"].min()),
      "to", round(quakes["depth"].max()), "km")

The ridges are uniformly pale: everything that happens at a spreading centre happens in the top
few tens of kilometres. The dark dots — the deep ones — appear in only a handful of places, and
every one of them is a place where the previous figure drew a trench: South America, Japan,
Indonesia, Tonga. Notice also what is *not* dark. The black line across Asia has no dark dots
anywhere along it, because the Himalaya and the Zagros are collisions with no plate going down —
the caution from the last section, visible.

And in the places that do go dark, the dark dots are not on the trench line; they are set back
from it. That offset is the whole point, and it is easier to measure on one arc than on the whole
world.

## One arc, close up

Zooming a map means changing `plt.xlim` and `plt.ylim` and nothing else — the data is the same
data. To keep only the earthquakes inside the box, filter on latitude and longitude the way you
filtered on magnitude in the tables week, joining the four conditions with `&`, one bracketed
condition at a time.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
quakes = quakes[quakes["type"] == "earthquake"]
trenches = boundaries[boundaries["kind"] == "trench"]

### ✏️ Your turn 3

Take South America: latitude -45 to 5, longitude -85 to -60.

1. Filter `quakes` to that box and call it `south`.
2. Draw it exactly like your depth map — `c=south["depth"]`, `cmap="plasma_r"`, `vmin=0`,
   `vmax=600`, a colorbar — but with the limits set to the box, and with the trenches drawn on top
   in black (`plt.plot(trenches.lon, trenches.lat, color="black", lw=1.2)`).
3. Then print two numbers: the median longitude of the shallow events (`depth` at most 70) and the
   median longitude of the deep ones (`depth` over 300).

**Use these names**, because the self-check looks for them: `south`, `shallow_lon`, `deep_lon`.

In [ ]:
# ← your answer here


assert len(south) > 0, "no earthquakes in the box — check the four limits"
print("✓ South America —", len(south), "earthquakes; the deep ones are",
      round(deep_lon - shallow_lon, 1), "degrees of longitude further east")

The black trench line runs down the coast; the pale shallow events sit on it, and the dark ones
sit in two tight clusters several degrees inland. Degrees of longitude are hard to feel, so turn
them into kilometres. One degree of longitude is the equator's circumference divided by 360, times
`cos(latitude)`. Earth's mean radius is 6371 km — the value the International Union
of Geodesy and Geophysics publishes, read from `en.wikipedia.org/wiki/Earth_radius` on 2026-08-31.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
south = quakes[(quakes["latitude"] >= -45) & (quakes["latitude"] <= 5)
               & (quakes["longitude"] >= -85) & (quakes["longitude"] <= -60)]
shallow_lon = south[south["depth"] <= 70]["longitude"].median()
deep_lon = south[south["depth"] > 300]["longitude"].median()

In [ ]:
deep_lat = south[south["depth"] > 300]["latitude"].median()
km_per_degree = 2 * np.pi * 6371 / 360 * np.cos(np.deg2rad(deep_lat))

print("one degree of longitude here:", round(km_per_degree), "km")
print("the deep events sit", round((deep_lon - shallow_lon) * km_per_degree), "km inland")
print("the deepest event in this box:", round(south["depth"].max()), "km")

About 907 kilometres. Put that beside the depths and the shape is forced: the
earthquakes get deeper the further inland you go, from a few tens of kilometres at the coast to
648 km under the interior of the continent. They are not scattered through the
mantle at random depths; they lie on a surface that starts at the trench and dips away under the
continent. That surface is the plate: cold ocean floor bending down at the trench and sliding into
the mantle, with the deep earthquakes happening inside it.

That is *where*, and it is only *where*. Being cold is why the slab is the only place down there
that makes earthquakes at all — it is the one cold thing in a mantle that is otherwise hot and
flowing quietly — but being cold does not make deep rock breakable. Six hundred kilometres down
the weight of the rock above is about 21 gigapascals, 210,000 times the
pressure of the air in this room (worked out from PREM, the standard profile of density inside the
Earth, `ds.iris.edu/files/products/emc/data/PREM/PREM_1s.csv`, read 2026-08-31), and under that
squeeze rock does not crack open however cold it is. So the strangeness this section opened with
is real, and it is still unsolved. Seismologists have candidates: water, given up as minerals in
the slab break down, which can let a fault slip somewhere between 70 and 300 km; and below roughly
400 km, a mineral that has survived too long in the wrong crystal form flipping suddenly into a
denser one, or a thin sliding layer heating itself faster than the heat can escape. Which of them
does the work, and at what depth, is still argued about (Green and Houston, *Annual Review of Earth
and Planetary Sciences*, 1995; Houston, *Deep Earthquakes*, in the *Treatise on Geophysics*, 2015).
Your figure cannot tell you the mechanism. It tells you the geometry, which is the part that is
settled: whatever is breaking down there, it is breaking inside a sinking plate.

## How big is big

That is where; the rest of the notebook is how big. Volcanologists score an eruption on the
**Volcanic Explosivity Index**, a whole number from 0 to 8, and the eruption table carries it in
`ExplosivityIndexMax`. There are 11,089 eruptions in the file — every one known
since the last ice age.

### Predict before you run

VEI 0 is the smallest kind of eruption and VEI 2 is two steps up. The record holds
4,030 eruptions at VEI 2. How many do you think it holds at VEI 0? Write your guess into
`my_guess` before you run the cell.

In [ ]:
my_guess = 20000

vei_counts = eruptions["ExplosivityIndexMax"].value_counts().sort_index()
print(vei_counts)
print("you guessed", my_guess, "at VEI 0; the record holds", int(vei_counts.loc[0]))

Fewer, not more — 1,019 against 4,030. Hold that; it is the second surprise of the
day, and we come back to it before the section is out.

First, what the index means. VEI is not a volume, it is an *index*: each whole step stands for
roughly ten times more erupted rock. From VEI 2 upwards the convention is that VEI *n* means at
least 10 to the power (*n* − 5) cubic kilometres — so VEI 5 is 1 km³ and VEI 7 is 100 km³.
(Newhall and Self defined the index in 1982; the thresholds here were read from
`en.wikipedia.org/wiki/Volcanic_explosivity_index` on 2026-08-31. The rule breaks below VEI 2,
where the steps are not tenfold, which is one reason not to trust the bottom of this scale.)

### ✏️ Your turn 4

Write `vei_volume(vei)`: one argument, a docstring saying what it does, and it returns the
smallest erupted volume in cubic kilometres that the index stands for — 10 to the power
(`vei` − 5).

Then print the volume for Tambora, which is VEI 7, the volume for Mount St Helens in 1980, which
is VEI 5, and how many times bigger the first is than the second.

**Use these names**, because the self-check looks for them: `vei_volume`.

In [ ]:
# ← your answer here


assert vei_volume(6) / vei_volume(5) == 10, "one step of VEI should be a factor of ten"
print("✓ VEI is an index, not a volume — two steps up is a factor of",
      round(vei_volume(7) / vei_volume(5)))

A factor of 100 between two eruptions that are only two apart on the scale. A count
of eruptions per VEI has the same problem: the classes at the top are rare by exactly as much as
they are big. Draw it on an ordinary axis and see what happens.

`plt.bar(positions, heights)` draws one bar per category — the right chart when the thing on the
bottom axis is a label rather than a measurement. `plt.subplot(1, 2, 1)` means *one row of two
panels, and I am drawing in the first*, so two charts can sit side by side and be compared.

In [ ]:
plt.subplot(1, 2, 1)
plt.bar(vei_counts.index, vei_counts.values, color="darkorange")
plt.xlabel("VEI")
plt.ylabel("number of eruptions")
plt.title("ordinary axis, n = " + str(int(vei_counts.sum())))

plt.subplot(1, 2, 2)
plt.bar(vei_counts.index, vei_counts.values, color="darkorange")
plt.yscale("log")
plt.xlabel("VEI")
plt.ylabel("number of eruptions")
plt.title("log axis, n = " + str(int(vei_counts.sum())))
plt.show()

On the left, everything from VEI 4 up is a flat line on the floor. There are 7 VEI 7
eruptions in the record and the chart cannot show you that there are any at all. On the right, the
same numbers on a log axis: *when the values span factors of a thousand, plot the exponents
instead and a curve becomes a line.* Every class is now readable, and the tops of the bars from
VEI 2 to VEI 6 come down in near-equal steps — which on a log axis means each class is a roughly
constant factor rarer than the one below it.

And now the low end is impossible to miss. VEI 1 and VEI 0 sit *below* VEI 2, which is the wrong
way round. The steps from VEI 2 upwards say that going one step *down* the scale should multiply
the count, so the two smallest classes ought to be the two tallest bars on the chart, and instead
they are shorter than the class above them. Either the world really does make fewer small
eruptions than middling ones, or the record is missing them. There is a way to tell the two apart:
look at a window of the catalogue where the recording is better.

### ✏️ Your turn 5

Modern volcano monitoring is nothing like the record of the last ten thousand years as a whole. So
cut the table to eruptions that started in 1950 or later, using `StartDateYear`, and draw the same
log-axis bar chart for that window alone.

Then, to compare the windows as numbers rather than pictures, loop over
`first_years = [-60000, 1800, 1950]`. For each one, take the eruptions from that year onwards,
count them by VEI with `.value_counts()`, and print how many VEI 1 eruptions there are for each
VEI 2 — `counts.loc[1] / counts.loc[2]`, rounded to two decimals.

**Use these names**, because the self-check looks for them: `recent`, `recent_counts`.

In [ ]:
# ← your answer here


assert len(recent) < len(eruptions), "recent should be a slice of the table, not all of it"
print("✓ the modern window —", len(recent), "eruptions since 1950, of which",
      int(recent_counts.sum()), "carry a VEI")

The ratio climbs from 0.36 over the whole record to 0.87 since 1950. As
the recording gets better the small eruptions come back, which settles it: the missing bars are a
property of the archive, not of the planet. A VEI 2 eruption a thousand years ago left a layer of
ash somebody can still dig up. A VEI 0 eruption a thousand years ago left nothing, unless somebody
was standing there.

Earthquake magnitude works the same way, and its constant is worth knowing. The energy an
earthquake radiates as seismic waves goes as log₁₀ *E* = 1.5 *M* + 4.8, with *E* in joules — the
Gutenberg–Richter convention; the factor of about 31.6 in energy per whole magnitude step was read
from `en.wikipedia.org/wiki/Richter_scale` on 2026-08-31.

In [ ]:
def quake_energy(mag):
    """The energy an earthquake of this magnitude radiates as seismic waves, in joules."""
    return 10 ** (1.5 * mag + 4.8)


print("an M9 against an M6:", round(quake_energy(9) / quake_energy(6)), "times the energy")

energy = quake_energy(quakes["mag"])
print(quakes.sort_values("mag", ascending=False).head(1)[["mag", "place"]])
print("that one event's share of the catalogue's energy:", round(energy.max() / energy.sum(), 3))

One earthquake out of 12,847 released 18% of the seismic
energy of twenty-six years. Both scales — VEI and magnitude — are built that way on purpose: they
are indices with a factor hiding in every step, so a count of them belongs on a log axis and a
difference of two on the scale is never a difference of two in the world.

## The question, answered

**On the plate boundaries, and the deep ones behind the trenches.** Every earthquake and almost
every volcano you plotted sits on the edge of a tectonic plate: the earthquakes drew the boundary
lines before you loaded them, including the mid-Atlantic ridge, where no land marks it at all.
69% of the volcanoes sit at subduction zones, where water carried down with the
sinking plate melts the mantle above it. And the deep earthquakes, absent from the ridges and
clustered near the trenches, sit 907 km inland of the shallow ones in South America,
because they are happening inside a cold plate that is still sinking, hundreds of kilometres past
the point where it went under. How rock manages to break at all under that much pressure is a
question seismologists have not closed; *where* it breaks, your own figure settled. One arc is one
arc; the homework asks you to check a second one.

## Week 4 summary

**The question.** Where do earthquakes and volcanoes happen — and why there?

### What to remember

| | |
|---|---|
| **1** | Earthquakes and volcanoes sit on plate boundaries; the deep earthquakes sit behind the trenches, on subducting slabs. |
| **2** | A map is a scatter plot of longitude against latitude — no map library required. |
| **3** | Magnitude and VEI are both logarithmic, so plot them on log axes or the figure lies. |

### Code you met this week

| Function | What it does |
|---|---|
| `plt.subplot(rows, cols, k)` | draw several panels in one figure; k picks which one |
| `plt.bar(positions, heights)` | one bar per category — the right chart when the bottom axis is a label |
| `plt.yscale("log")` | count the axis in factors of ten, so classes a thousandfold apart are all readable |
| `plt.scatter(x, y, c=values, cmap=...)` | colour every dot by a third column, with vmin/vmax fixing what the ends mean |
| `plt.colorbar(label=...)` | the key that says what the colours mean |
| `counts.index / counts.values` | the labels and the numbers value_counts() handed back |
| `counts.sort_index()` | put the counts back in order of the thing counted |
| `column.str.startswith("Sub")` | a mask marking the rows whose text begins that way |

## Homework

Three parts, on the same two catalogues you already have loaded. If you have restarted since
class, run the setup cell at the top first and then the `type` filter in the first section.

### ✏️ Your turn 6

Class found that the eruption record is missing its smallest eruptions. Is the earthquake
catalogue missing its smallest earthquakes in the same way? The same picture answers it.

VEI came ready-made in whole numbers, so `.value_counts()` was enough. Magnitude does not, so bin
it first, exactly as you binned elevations in the grids week:

```
edges = np.arange(5.5, 9.6, 0.5)
counts, edges = np.histogram(quakes["mag"], bins=edges)
centres = (edges[:-1] + edges[1:]) / 2
```

Then draw `counts` against `centres` as a bar chart with `width=0.45`, put the count axis on a log
scale, label both axes, title it with the sample size, and print `counts` so you can read the
numbers off.

**Use these names**, because the self-check looks for them: `counts`, `centres`.

In [ ]:
# ← your answer here


assert counts.sum() == len(quakes), "every earthquake should land in a bin — check your edges"
print("✓ magnitudes on a log axis — the smallest bin holds", counts[0],
      "earthquakes and the next one", counts[1])

### ✏️ Your turn 7

Class drew South America from above. Seen from the side, the same scatter becomes a cross-section:
put longitude on the bottom axis and depth up the side, and the sinking plate draws itself.

**Choose one arc**, and say in a comment which you chose:

- **Chile** — latitude -32 to -14, longitude -80 to -58
- **Japan** — latitude 30 to 45, longitude 128 to 148

Filter `quakes` to your box and call it `arc`. Plot `arc["longitude"]` against `-arc["depth"]` —
the minus sign turns a depth into a height, so the picture is the right way up and the axis label
should say so. Label both axes and title the figure with the arc's name and how many earthquakes
are in it. Then work out `shallow_lon` and
`deep_lon`, the median longitudes of the events no deeper than 70 km and of those deeper than
300 km — the same two numbers you printed for South America.

The two boxes do not give the same answer, and both are right.

**Use these names**, because the self-check looks for them: `arc`, `shallow_lon`, `deep_lon`.

In [ ]:
# ← your answer here


assert len(arc) > 0, "no earthquakes in that box — check the four numbers"
if deep_lon > shallow_lon:
    side = "east"
else:
    side = "west"
print("✓ the slab —", len(arc), "earthquakes; the deep ones sit",
      round(abs(deep_lon - shallow_lon), 1), "degrees", side, "of the shallow ones")

### ✏️ Your turn 8

Two of the pictures in this notebook are counts on a log axis: the eruptions by VEI, and your own
magnitudes from part 6. One of them falls off at its low end and the other does not.

In three or four sentences, and using your own printed numbers — the two shortest VEI bars from
class, and the first two numbers in your `counts` array — say which chart has the broken low end,
and explain what is different about how the two catalogues were made. Your answer should say what
would have to be true for the *other* chart's low end to break as well.

*(Double-click this cell and replace this line with your answer.)*